# Exploratory Data Analysis (EDA).
Explore and understand the CheXpert_small and TBX11K datasets before training models.
#### Rationale:
1. Varify dataset contents, structure, and class distributions.
2. Detect potential imbalances or inconsistencies that may bias results.
3. Document image resolutions and formats to justify preprocessing choices.
4. Produce figures (class balance plots, histograms) to be discused in the report.

In [1]:
# Define dataset root paths (This notebook lives in notebooks/, so step one level up to reach the project root).
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# Define dataset paths relative to the project structure.
CHEXPERT_PATH = PROJECT_ROOT / "data" / "chexpert"
TBX11K_PATH   = PROJECT_ROOT / "data" / "tbx11k"

print('Project root:', PROJECT_ROOT)
print('CheXpert path exists:', CHEXPERT_PATH.exists())
print('TBX11K path exists:', TBX11K_PATH.exists())

Project root: /lab/px/msc-sept-24-ft-cohort-capstone-chrisanich
CheXpert path exists: True
TBX11K path exists: True


## 1. Dataset Overview.
Count images and inspect folder structure for CheXpert_small and TBX11K.
#### Rationale:
- Verify datasets were correctly downloaded and extracted.
- Get a first look at class balance (Normal, Pneumonia, Tuberculosis).
- Ensure consistency accross datasets before unifying them for training.
- Document findings with numbers to support the Methods section of the report.

In [2]:
# Count the number of image files in each dataset directory.
# This ensures that data is complete before further analysis.

def count_images_in_folder(path: Path, exts=(".jpg", ".jpeg", ".png")):
    """Count image files recursively in a folder."""
    return sum(1 for f in path.rglob("*") if f.suffix.lower() in exts)

chexpert_count = count_images_in_folder(CHEXPERT_PATH)
tbx11k_count = count_images_in_folder(TBX11K_PATH)

print('Image counts:')
print(f'- CheXpert_small: {chexpert_count:,} images')
print(f'- TBX11B: {tbx11k_count:,} images')

Image counts:
- CheXpert_small: 223,649 images
- TBX11B: 12,279 images


## Class Distribution (TBX11K).
Load the TBX11K dataset split list (`all_trainval.txt`) and infer labels from folder and filename conventions.
#### Rationale:
- TBX11K does not provide a ready-made CSV with labels.
- Instead, image labels are encoded in file paths (`sick/`, `tb/`, or filenames ending in `_1.png` for Tuberculosis (TB), others as Normal).
- Parsing the official split list ensures reproducibility and consistency with the dataset authors' organisation.
- Class counts are essential to detect imbalance before modeling training.

In [7]:
import pandas as pd

def load_tbx11k_from_list(list_file):
    """
    Load a TBX11K split list and assign class labels.

    Parameters
    ----------
    list_file : str or Path
        Path to a test file (e.g., all_train.txt, all_val.txt, all_test.txt, all_trainval.txt).
        Each line contains a relative path to an image file.

    Returns
    -------
    pd.DataFrame
        DataFrame with two columns:
        - 'Path': relative image path
        - "Class': assigned label ('TB' or 'Normal')
    """
    with open(list_file, "r") as f:
        paths = [line.strip() for line in f if line.strip()]

    records = []
    for p in paths:
        # Label assignment is based on TBX11K naming conventions.
        if p.startswith(("sick", "tb")) or p.endswith("_1.png"):
            label = "TB"
        else:
            label = "Normal"
        records.append({"Path": p, "Class": label})

    return pd.DataFrame(records)

# Load the combined training plus validation split.
tbx11k_df = load_tbx11k_from_list(TBX11K_PATH / "lists" / "all_trainval.txt")

# Summarise class distribution.
print("TBX11K class distribution (all_trainval split):")
print(tbx11k_df["Class"].value_counts())

TBX11K class distribution (all_trainval split):
Class
TB        4798
Normal    4178
Name: count, dtype: int64
